In [12]:
import pandas as pd
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.ingestion.ckan_client import (
    get_dataset,
    get_resources,
    find_resources_by_year
)

f:\Silvio\Proyectos Ingenieria de datos\Argentina-oil-gas-data-platform


In [ ]:
DATASET_ID = "produccion-de-petroleo-y-gas-por-pozo"

dataset = get_dataset(DATASET_ID)
resources = get_resources(dataset)

resources_2026 = find_resources_by_year(resources, 2026)

for resource in resources_2026:
    print(f"Nombre: {resource.get('name')}")
    print(f"Formato: {resource.get('format')}")
    print(f"URL: {resource.get('url')}")
    print("-" * 80)

In [ ]:
for resource in resources_2026:
    print(f"Cargando: {resource.get('name')}")
    df_produccion_2026 = pd.read_csv(
    resources_2026[0]["url"])
    df_ddjj_2026 = pd.read_csv(
    resources_2026[1]["url"])

print("Producción 2026:", df_produccion_2026.shape)
print("DDJJ 2026:", df_ddjj_2026.shape)

In [ ]:
display(df_produccion_2026.head())
display(df_ddjj_2026.head())
print("Columnas Producción 2026:")
print(df_produccion_2026.columns.tolist())

print("\nColumnas DDJJ 2026:")
print(df_ddjj_2026.columns.tolist())

In [ ]:
df_produccion_2026[
    ["idempresa", "anio", "mes", "idpozo"]
].duplicated().sum()

df_ddjj_2026[
    ["idempresa", "anio", "mes", "idpozo"]
].duplicated().sum()

In [ ]:
key_cols = ["idempresa", "anio", "mes", "idpozo"]

prod_keys = df_produccion_2026[key_cols].drop_duplicates()
ddjj_keys = df_ddjj_2026[key_cols].drop_duplicates()

print("Claves únicas Producción:", len(prod_keys))
print("Claves únicas DDJJ:", len(ddjj_keys))

prod_key_set = set(
    map(tuple, prod_keys.to_numpy())
)

ddjj_key_set = set(
    map(tuple, ddjj_keys.to_numpy())
)

only_prod = prod_key_set - ddjj_key_set
only_ddjj = ddjj_key_set - prod_key_set

print("Solo en Producción:", len(only_prod))
print("Solo en DDJJ:", len(only_ddjj))

In [ ]:
df_produccion_2026.info()
df_ddjj_2026.info()

In [ ]:
df_produccion_2026[
    ["prod_pet", "prod_gas", "prod_agua", "iny_agua", "iny_gas", "iny_co2"]
].describe()

In [ ]:
production_cols = [
    "prod_pet",
    "prod_gas",
    "prod_agua",
    "iny_agua",
    "iny_gas",
    "iny_co2",
    "iny_otro"
]

for col in production_cols:
    print(
        f"{col:12} | "
        f"ceros: {(df_produccion_2026[col] == 0).sum():>8,} | "
        f"no ceros: {(df_produccion_2026[col] != 0).sum():>8,}"
    )

only_ddjj_df = df_ddjj_2026.merge(
    df_produccion_2026[
        ["idempresa", "anio", "mes", "idpozo"]
    ],
    on=["idempresa", "anio", "mes", "idpozo"],
    how="left",
    indicator=True
)

only_ddjj_df = only_ddjj_df[
    only_ddjj_df["_merge"] == "left_only"
]

print(f"Registros solo en DDJJ: {len(only_ddjj_df):,}")

display(
    only_ddjj_df[
        [
            "idempresa",
            "anio",
            "mes",
            "idpozo",
            "prod_pet",
            "prod_gas",
            "prod_agua",
            "tipoestado",
            "tipopozo",
            "rectificado",
            "habilitado",
            "fechaingreso",
            "fecha_data"
        ]
    ].head(20)
)

## 4. Investigación de diferencias entre recursos

El recurso DDJJ contiene 1.563 registros que no aparecen en el recurso Producción utilizando como clave candidata:

`idempresa + anio + mes + idpozo`

Se investigará la naturaleza de estos registros antes de seleccionar la fuente principal del pipeline.

In [ ]:
key_cols = ["idempresa", "anio", "mes", "idpozo"]

only_ddjj_df = df_ddjj_2026.merge(
    df_produccion_2026[key_cols],
    on=key_cols,
    how="left",
    indicator=True
)

only_ddjj_df = only_ddjj_df[
    only_ddjj_df["_merge"] == "left_only"
].copy()

print(f"Registros solo en DDJJ: {len(only_ddjj_df):,}")

In [ ]:
display(
    only_ddjj_df[
        [
            "idempresa",
            "anio",
            "mes",
            "idpozo",
            "prod_pet",
            "prod_gas",
            "prod_agua",
            "tipoestado",
            "tipopozo",
            "rectificado",
            "habilitado",
            "fechaingreso",
            "fecha_data"
        ]
    ].head(20)
)

In [ ]:
only_ddjj_df["mes"].value_counts().sort_index()
only_ddjj_df["tipoestado"].value_counts(dropna=False)
only_ddjj_df["rectificado"].value_counts(dropna=False)
only_ddjj_df["habilitado"].value_counts(dropna=False)


In [ ]:
only_ddjj_df["idempresa"].value_counts().head(20)
only_ddjj_df["provincia"].value_counts(dropna=False)
only_ddjj_df["cuenca"].value_counts(dropna=False).head(20)
only_ddjj_df["tipo_de_recurso"].value_counts(dropna=False)

In [ ]:
only_ddjj_df[
    ["anio", "mes", "fecha_data", "fechaingreso"]
].head(20)
only_ddjj_df["fecha_data"].value_counts()
only_ddjj_df["fechaingreso"].value_counts().head(20)


In [ ]:
print("Producción:")
print(df_produccion_2026["fecha_data"].min())
print(df_produccion_2026["fecha_data"].max())

print("\nDDJJ:")
print(df_ddjj_2026["fecha_data"].min())
print(df_ddjj_2026["fecha_data"].max())

In [ ]:
compare_cols = [
    "idempresa",
    "anio",
    "mes",
    "idpozo",
    "prod_pet",
    "prod_gas",
    "prod_agua",
    "iny_agua",
    "iny_gas",
    "iny_co2",
    "iny_otro",
    "tipoestado",
    "rectificado",
    "habilitado"
]
comparison = df_produccion_2026[compare_cols].merge(
    df_ddjj_2026[compare_cols],
    on=["idempresa", "anio", "mes", "idpozo"],
    how="inner",
    suffixes=("_prod", "_ddjj")
)

print(f"Registros con la misma clave: {len(comparison):,}")

In [ ]:
production_columns = [
    "prod_pet",
    "prod_gas",
    "prod_agua",
    "iny_agua",
    "iny_gas",
    "iny_co2",
    "iny_otro"
]

for col in production_columns:
    differences = (
        comparison[f"{col}_prod"] != comparison[f"{col}_ddjj"]
    ).sum()

    print(f"{col}: {differences:,} diferencias")

In [ ]:
key_cols = ["idempresa", "anio", "mes", "idpozo"]

comparison = df_produccion_2026.merge(
    df_ddjj_2026,
    on=key_cols,
    how="inner",
    suffixes=("_prod", "_ddjj")
)

print(f"Registros comparados: {len(comparison):,}")

In [ ]:
common_columns = [
    col for col in df_produccion_2026.columns
    if col not in key_cols
]

print(f"Columnas a comparar: {len(common_columns)}")
print(common_columns)

In [ ]:
metadata_columns = [
    "tipoextraccion",
    "tipoestado",
    "tipopozo",
    "observaciones",
    "fechaingreso",
    "rectificado",
    "habilitado",
    "idusuario",
    "empresa",
    "sigla",
    "formprod",
    "profundidad",
    "formacion",
    "idareapermisoconcesion",
    "areapermisoconcesion",
    "idareayacimiento",
    "areayacimiento",
    "cuenca",
    "provincia",
    "tipo_de_recurso",
    "proyecto",
    "clasificacion",
    "subclasificacion",
    "sub_tipo_recurso",
    "fecha_data"
]
for col in common_columns:
    prod_values = comparison[f"{col}_prod"]
    ddjj_values = comparison[f"{col}_ddjj"]

    differences = (
        ~(
            prod_values.eq(ddjj_values)
            | (prod_values.isna() & ddjj_values.isna())
        )
    ).sum()

    print(f"{col}: {differences:,} diferencias")

In [ ]:
production_columns = [
    "prod_pet",
    "prod_gas",
    "prod_agua",
    "iny_agua",
    "iny_gas",
    "iny_co2",
    "iny_otro"
]

only_ddjj_df[production_columns].describe()

In [ ]:
only_ddjj_df[production_columns].sum()

In [ ]:
only_ddjj_df[
    production_columns
].ne(0).any(axis=1).value_counts()

In [ ]:
only_ddjj_df.groupby(
    ["anio", "mes"]
).size()

In [ ]:
only_ddjj_df.groupby(
    ["empresa", "provincia"]
).size().sort_values(ascending=False)

In [19]:
import importlib
import src.ingestion.ckan_client as ckan_client

importlib.reload(ckan_client)

from src.ingestion.ckan_client import (
    get_dataset,
    get_resources,
    find_resources_by_year,
    find_resource_by_name,
    download_resource
)


In [21]:
production_resource = find_resource_by_name(
    resources_2026,
    "Producción de Pozos de Gas y Petróleo – 2026"
)

print(production_resource["name"])
print(production_resource["format"])

production_resource = find_resource_by_name(
    resources_2026,
    "Producción de Pozos de Gas y Petróleo – 2026"
)

print(f"Nombre: {production_resource['name']}")
print(f"Formato: {production_resource['format']}")
print(f"URL: {production_resource['url']}")

Producción de Pozos de Gas y Petróleo – 2026
CSV
Nombre: Producción de Pozos de Gas y Petróleo – 2026
Formato: CSV
URL: http://datos.energia.gob.ar/dataset/c846e79c-026c-4040-897f-1ad3543b407c/resource/fb7a47a0-cba9-4667-a004-6f6c1c346c23/download/produccin-de-pozos-de-gas-y-petrleo-2026.csv
